In [ ]:
## import libraries

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text
import psycopg2
import sklearn

## create database engine using postgresql connection string:
USERNAME = "postgres"
PASSWORD = "******"
HOST = "localhost"
PORT = "5432"
DB_NAME = "fraud_capstone"

engine = create_engine(f"postgresql+psycopg2://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}")

query = """
SELECT * FROM staging_schema.cleaned_transactions
ORDER BY time;
"""

df = pd.read_sql(query, engine)

df = df.sort_values("time")

In [ ]:
#rolling average

df["rolling_avg_amount_10"] = df["amount"].rolling(10).mean()

df.head()


In [ ]:
#rolling standard deviation

df["rolling_std_amount_10"] = df["amount"].rolling(10).std()

df.head()


In [ ]:
#rolling sum

df["rolling_sum_amount_10"] = df["amount"].rolling(10).sum()

df.head()


In [ ]:
#z-score

df["amount_zscore"] = (
(df["amount"] - df["amount"].mean()) /
df["amount"].std()
)

df.head()


In [ ]:
#high value flag

threshold = df["amount"].quantile(0.95)

df["high_value_flag"] = df["amount"] > threshold

df.head()


In [ ]:
#amount velocity

df["amount_velocity"] = df["amount"].diff()

df.head()

In [ ]:
#drop rows generated by the rolling windows

df = df.dropna()

#load into curated table.

df.to_sql(
"feature_engineered_transactions",
engine,
schema="curated_schema",
if_exists="replace",
index=False
)
